In [ ]:
import json
from os import path

import geopandas as gpd
import pandas as pd
import seaborn as sns
import xarray as xr
import xvec  # noqa: F401
from skops import io

SKOPS_TRUSTED = "scipy.interpolate._bsplines.BSpline"

sns.set_style("whitegrid")

In [ ]:
aws_ts_cube_filepath = "../data/interim/bern-aws-ts-cube.nc"
lcd_ts_df_filepath = "../data/interim/bern-lcd-ts-df.csv"
lcd_stations_gdf_filepath = "../data/interim/bern-lcd-stations.gpkg"

# model_filepath = f"../models/{station_model}-gam-ar.pickle"
models_dir = "../models"
station_model = "Decentlab"
station_model_dict_filepath = "../data/processed/station-model-dict.json"
station_scale_dict_filepath = "../data/processed/station-scale-dict.json"

dst_ts_df_filepath = "../data/interim/bern-cor-ts-df.csv"

In [ ]:
# aws_ts_df = pd.read_csv(aws_ts_df_filepath, parse_dates=["time"]).set_index("time")
lcd_ts_df = pd.read_csv(lcd_ts_df_filepath, index_col="time", parse_dates=["time"])
lcd_ts_df.head()

,1,10,101,102,111,112,113,116,117,124,...,83,86,87,89,9,98,99,134,135,52
time,,,,,,,,,,,,,,,,,,,,,
2023-06-01 00:00:00,13.245212,13.936383,NaN,13.608377,13.985339,12.842438,NaN,12.282114,10.598026,13.898553,...,13.591020,13.226965,12.413405,14.421047,11.615867,11.760510,12.112103,NaN,NaN,NaN
2023-06-01 01:00:00,12.611009,13.336894,NaN,13.129943,13.643091,12.456130,NaN,11.706658,10.223735,13.460174,...,13.403652,12.958597,12.059141,14.175822,11.211312,11.408026,11.638120,NaN,NaN,NaN
2023-06-01 02:00:00,12.210460,12.980850,NaN,12.857570,13.356921,12.036889,NaN,11.239796,10.299840,13.309300,...,13.054284,12.549147,11.878004,13.448157,10.942054,11.149449,11.464548,NaN,NaN,NaN
2023-06-01 03:00:00,12.405839,12.821520,NaN,12.674652,13.068526,11.904707,NaN,10.979438,10.421340,12.924328,...,12.767669,12.190878,11.756504,13.238092,10.754686,10.793406,11.027949,NaN,NaN,NaN
2023-06-01 04:00:00,12.718713,12.669757,NaN,12.507757,12.944355,12.266982,12.611454,10.997241,10.390631,12.494405,...,12.864246,12.071603,11.703543,13.471300,11.058658,11.222438,11.245581,NaN,NaN,NaN


In [ ]:
lcd_stations_gdf = gpd.read_file(lcd_stations_gdf_filepath).set_index("station_id")
lcd_stations_gdf.head()

,NAME_old,NAME_new,NORD_CHTOPO,OST_CHTOPO,LV_03_E,LV_03_N,HüM,geometry
station_id,,,,,,,,
1,VonRoll PH,VonRoll,46.95291,7.42252,2598773.7,1200203.5,553.9,POINT (2598773.471 1200203.276)
10,"Länggasse, Neufeldstrasse 135",Hintere Laenggasse,46.95789,7.43541,2599754.9,1200757.0,558.5,POINT (2599754.731 1200756.774)
101,Steinhölzliwald,Steinhoelzliwald,46.93514,7.42805,2599194.5,1198227.6,573.7,POINT (2599194.143 1198227.698)
102,"Vorderes LG-Quartier, Falkenweg 5",Vordere Laenggasse,46.95171,7.43714,2599886.1,1200069.3,561.6,POINT (2599886.391 1200069.733)
111,Tellstrasse,Tellstrasse,46.96017,7.46146,2601737.6,1201010.4,556.1,POINT (2601737.49 1201010.468)


In [ ]:
aws_ts_cube = xr.open_dataset(aws_ts_cube_filepath).xvec.decode_cf()
aws_ts_cube

<xarray.Dataset> Size: 106kB
Dimensions:              (geometry: 1, time: 4416)
Coordinates:
  * geometry             (geometry) object 8B POINT (2601934 1204410)
  * time                 (time) datetime64[ns] 35kB 2023-06-01 ... 2024-08-31...
    station_id           (geometry) <U3 12B ...
Data variables:
    temperature          (time, geometry) float64 35kB ...
    radiation_shortwave  (time, geometry) float64 35kB ...
Indexes:
    geometry  GeometryIndex (crs=EPSG:2056)

In [ ]:
with open(station_model_dict_filepath) as src:
    model_filename = json.load(src)[station_model]
    model = io.load(path.join(models_dir, model_filename), trusted=SKOPS_TRUSTED)

with open(station_scale_dict_filepath) as src:
    scale = json.load(src)[station_model]

In [ ]:
window = pd.Timedelta(minutes=scale)
# TODO: get frequency from the time dimension of the data cube
ref_freq = pd.Timedelta("1 hour")
window_len = int(window / ref_freq)

# for each LCD, compute the previous radiation
X_df = (
    aws_ts_cube["radiation_shortwave"]
    .rolling(time=window_len, min_periods=window_len)
    .sum()
    .set_index(geometry="station_id")
    .rename(geometry="station_id")
    .to_pandas()
    .dropna(how="all", axis="columns")
    .dropna()
)
ref_station_gdf = aws_ts_cube[["geometry"]].xvec.to_geodataframe()
ref_station_gdf = ref_station_gdf[ref_station_gdf["station_id"].isin(X_df.columns)]

# get a map of the nearest AWS for each LCD
nearest_aws_dict = lcd_stations_gdf.sjoin_nearest(ref_station_gdf)[
    "station_id_right"
].to_dict()

In [ ]:
yhat_df = pd.DataFrame(
    {
        column: model.predict(
            X_df[[nearest_aws_dict[column]]].rename(
                columns={col: "radiation_shortwave" for col in X_df.columns}
            )
        )
        for column in lcd_ts_df.columns
    },
    index=X_df.index,
)
yhat_df.head()

,1,10,101,102,111,112,113,116,117,124,...,83,86,87,89,9,98,99,134,135,52
time,,,,,,,,,,,,,,,,,,,,,
2023-06-01 01:00:00,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,...,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199
2023-06-01 02:00:00,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,...,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199,-0.348199
2023-06-01 03:00:00,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,...,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194,-0.343194
2023-06-01 04:00:00,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,...,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755,-0.201755
2023-06-01 05:00:00,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,...,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577,0.210577


In [ ]:
lcd_ts_df.sub(yhat_df).dropna(how="all").to_csv(dst_ts_df_filepath)